In [0]:
#1. leemos el archivo JSON multilinea

# Importamos las librerias que se van a utilizar
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define el la estructura personName
language_role_schema = StructType(fields = [
    StructField("roleId", IntegerType(), True),
    StructField("languageRole", StringType(), True)
])

# Cargamos el archivo utilizando la estructura definida
language_role_df = spark.read\
    .schema(language_role_schema)\
    .option("multiLine", "true")\
    .json("abfss://bronze@lsdata01.dfs.core.windows.net/language_role.json")

# Mostramos el resultado
display(language_role_df)



In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas
from pyspark.sql.functions import col, concat, current_timestamp, lit
language_role_renamed_df = language_role_df\
    .withColumnRenamed("roleId", "role_id")\
    .withColumnRenamed("languageRole", "language_role")\
    .withColumn("ingestion_date", current_timestamp())\
    .withColumn("enviroment", lit("Produccion"))
    
display(language_role_renamed_df)


In [0]:
#Paso 4 - Guardar datos en datalake en formato parket 
language_role_renamed_df.write.mode("overwrite").parquet("abfss://silver@lsdata01.dfs.core.windows.net/language_role")
df = spark.read.parquet("abfss://silver@lsdata01.dfs.core.windows.net/language_role")
display(df)
